In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import matplotlib.pyplot as plt

In [2]:
class SineWaveDataset(Dataset):
    def __init__(self, sequence_length=10, num_samples=1000):
        self.sequence_length = sequence_length
        self.num_samples = num_samples
        self.data = self._generate_sine_wave_data()

    def _generate_sine_wave_data(self):
        x = np.linspace(0, 100, self.num_samples + self.sequence_length)
        y = np.sin(x)
        data = []
        for i in range(self.num_samples):
            seq = y[i:i + self.sequence_length]
            target = y[i + self.sequence_length]
            data.append((seq, target))
        return data

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        seq, target = self.data[idx]
        return torch.tensor(seq, dtype=torch.float32), torch.tensor(target, dtype=torch.float32)

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(RNNModel, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(1, x.size(0), hidden_size).to(x.device)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out

In [4]:
dataset = SineWaveDataset(sequence_length=10, num_samples=1000)

In [5]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [6]:
input_size = 1
hidden_size = 50
output_size = 1

In [8]:
model = RNNModel(input_size, hidden_size, output_size)
model

RNNModel(
  (rnn): RNN(1, 50, batch_first=True)
  (fc): Linear(in_features=50, out_features=1, bias=True)
)

In [9]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [10]:
num_epochs = 50

In [11]:
for epoch in range(num_epochs):
    for seq, target in dataloader:
        seq = seq.unsqueeze(-1)
        target = target.unsqueeze(-1)
        optimizer.zero_grad()
        output = model(seq)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [10/50], Loss: 0.0007
Epoch [20/50], Loss: 0.0003
Epoch [30/50], Loss: 0.0000
Epoch [40/50], Loss: 0.0000
Epoch [50/50], Loss: 0.0000


In [12]:
model.eval()
with torch.no_grad():
    test_seq = torch.tensor(np.sin(np.linspace(0, 10, 10)), dtype=torch.float32).unsqueeze(-1).unsqueeze(0)
    predicted = model(test_seq)
    print(f'Predicted: {predicted.item():.4f}, Actual: {np.sin(2.0):.4f}')

Predicted: 1.0850, Actual: 0.9093


In [13]:
test_seq = test_seq.squeeze().numpy()